In [ ]:
import os
from pathlib import Path

print("Current working directory:", os.getcwd())
print()
print("What's in the current directory:")
for item in Path(".").iterdir():
    print(" -", item.name)
print()
print("Does '../data/raw' exist from here?", (Path("..") / "data" / "raw").exists())
print("Does './data/raw' exist from here?", (Path(".") / "data" / "raw").exists())

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

DATA_DIR = Path("..") / "data" / "raw"

HOSPITAL_FILES = {
    "Baylor University Medical Center (Dallas)":
        "baylor_university_medical_center-69947_parsed.duckdb",
    "Methodist Dallas Medical Center":
        "methodist_dallas_medical_center-6000b_parsed.duckdb",
    "Parkland Health (Dallas)":
        "parkland_health-6e88d_parsed.duckdb",
    "Texas Health Presbyterian Hospital Plano":
        "texas_health_presbyterian_hospital_plano-6ad81_parsed.duckdb",
    "Medical City Alliance Hospital (Fort Worth)":
        "medical_city_alliance_hospital-77912_parsed.duckdb",
}
KNEE_MRI_CPT = "73721"
TARGET_PAYERS = ["Blue Cross", "UnitedHealthcare", "Aetna"]
for hospital, filename in HOSPITAL_FILES.items():
    path = DATA_DIR / filename
    status = "✓" if path.exists() else "✗ MISSING"
    print(f"{status} {hospital}: {filename}")

In [ ]:
baylor_path = DATA_DIR / HOSPITAL_FILES["Baylor University Medical Center (Dallas)"]
print(f"Connecting to: {baylor_path}")
print(f"File size: {baylor_path.stat().st_size / 1024 / 1024:.1f} MB")
print()

con = duckdb.connect(str(baylor_path), read_only=True)

tables = con.execute("SHOW TABLES").fetchdf()
print("Tables in this database:")
print(tables)

In [ ]:
schema = con.execute("DESCRIBE standard_charge_details").fetchdf()
print(f"Columns in standard_charge_details: {len(schema)} total")
print()
print(schema)

with pd.option_context('display.max_rows', None):
    print(schema)

In [ ]:
required_cols = [
    'cpt', 'description', 'gross_charge', 'setting',
    'payer_name', 'payer_group', 'payer_type', 'plan_name',
    'standard_charge_dollar', 'standard_charge_percentage'
]

actual_cols = set(schema['column_name'].tolist())

print("Column check:")
for col in required_cols:
    status = "✓" if col in actual_cols else "✗ MISSING"
    print(f"  {status} {col}")

In [ ]:

query = f"""
    SELECT
        description,
        setting,
        payer_name,
        payer_group,
        payer_type,
        standard_charge_dollar,
        gross_charge
    FROM standard_charge_details
    WHERE cpt = '{KNEE_MRI_CPT}'
    LIMIT 10
"""
result = con.execute(query).fetchdf()
print(f"Found {len(result)} rows (capped at 10)")
result

In [ ]:
q1 = "SELECT COUNT(*) AS total_rows, COUNT(cpt) AS rows_with_cpt FROM standard_charge_details"
print("--- Hypothesis 1: Is cpt populated? ---")
print(con.execute(q1).fetchdf())
print()

q2 = """
    SELECT cpt, COUNT(*) AS row_count
    FROM standard_charge_details
    WHERE cpt IN ('73721', '73722', '73723')
    GROUP BY cpt
"""
print("--- Hypothesis 2: Any knee MRI codes (73721/73722/73723)? ---")
print(con.execute(q2).fetchdf())
print()

q3 = """
    SELECT cpt, COUNT(*) AS row_count
    FROM standard_charge_details
    WHERE cpt LIKE '737%'
    GROUP BY cpt
    ORDER BY cpt
"""
print("--- Hypothesis 3: Any CPT codes starting with '737' (MRI lower extremity)? ---")
print(con.execute(q3).fetchdf())
print()

q4 = """
    SELECT DISTINCT description, cpt, hcpcs
    FROM standard_charge_details
    WHERE LOWER(description) LIKE '%knee%'
      AND (LOWER(description) LIKE '%mri%' OR LOWER(description) LIKE '%magnetic%')
    LIMIT 10
"""
print("--- Hypothesis 4: Descriptions containing 'knee' and 'MRI'/'magnetic'? ---")
print(con.execute(q4).fetchdf())

In [ ]:


qB = """
    SELECT DISTINCT description, cpt, hcpcs, rc
    FROM standard_charge_details
    WHERE LOWER(description) LIKE '%mri%'
       OR LOWER(description) LIKE '%magnetic resonance%'
    LIMIT 20
"""
print("--- B: Any MRI descriptions (any body part) ---")
print(con.execute(qB).fetchdf())
print()

qC = """
    SELECT cpt, description, setting, COUNT(*) as row_count
    FROM standard_charge_details
    WHERE cpt IS NOT NULL
    GROUP BY cpt, description, setting
    ORDER BY row_count DESC
    LIMIT 10
"""
print("--- C: Top 10 most common CPT-coded services ---")
print(con.execute(qC).fetchdf())

In [ ]:
query = f"""
    SELECT
        description,
        setting,
        cpt,
        hcpcs,
        rc,
        payer_name,
        payer_group,
        payer_type,
        standard_charge_dollar,
        gross_charge
    FROM standard_charge_details
    WHERE cpt = '{KNEE_MRI_CPT}'
       OR hcpcs = '{KNEE_MRI_CPT}'
    LIMIT 20
"""
result = con.execute(query).fetchdf()
print(f"Found {len(result)} rows (capped at 20)")
result

In [ ]:
query = f"""
    SELECT 
        payer_name,
        payer_group,
        payer_type,
        COUNT(*) AS row_count
    FROM standard_charge_details
    WHERE (cpt = '{KNEE_MRI_CPT}' OR hcpcs = '{KNEE_MRI_CPT}')
    GROUP BY payer_name, payer_group, payer_type
    ORDER BY row_count DESC
"""
result = con.execute(query).fetchdf()
print(f"Distinct payers offering knee MRI at Baylor: {len(result)}")
result

In [ ]:
query = f"""
    SELECT
        payer_group,
        payer_name,
        plan_name,
        setting,
        standard_charge_dollar,
        standard_charge_percentage,
        gross_charge
    FROM standard_charge_details
    WHERE (cpt = '{KNEE_MRI_CPT}' OR hcpcs = '{KNEE_MRI_CPT}')
      AND payer_group IN ('BCBS', 'UnitedHealthcare', 'Aetna')
    ORDER BY payer_group, standard_charge_dollar
"""
result = con.execute(query).fetchdf()
print(f"Knee MRI rates at Baylor for BCBS / UHC / Aetna: {len(result)} rows")
result

In [ ]:
query = f"""
    WITH distinct_rates AS (
        SELECT DISTINCT
            payer_group,
            payer_name,
            plan_name,
            standard_charge_dollar
        FROM standard_charge_details
        WHERE (cpt = '{KNEE_MRI_CPT}' OR hcpcs = '{KNEE_MRI_CPT}')
          AND payer_group IN ('BCBS', 'UnitedHealthcare', 'Aetna')
          AND standard_charge_dollar IS NOT NULL
    )
    SELECT
        payer_group,
        COUNT(*) AS plan_count,
        MIN(standard_charge_dollar) AS min_rate,
        MEDIAN(standard_charge_dollar) AS median_rate,
        MAX(standard_charge_dollar) AS max_rate,
        ROUND(AVG(standard_charge_dollar), 2) AS avg_rate
    FROM distinct_rates
    GROUP BY payer_group
    ORDER BY payer_group
"""
result = con.execute(query).fetchdf()
print("Baylor knee MRI summary (distinct plan-level rates):")
result

In [ ]:
con.close()

all_results = []

for hospital_name, filename in HOSPITAL_FILES.items():
    db_path = DATA_DIR / filename
    print(f"Querying: {hospital_name} ... ", end="")
    
    try:
        con = duckdb.connect(str(db_path), read_only=True)
        
        query = f"""
            WITH distinct_rates AS (
                SELECT DISTINCT
                    payer_group,
                    payer_name,
                    plan_name,
                    standard_charge_dollar
                FROM standard_charge_details
                WHERE (cpt = '{KNEE_MRI_CPT}' OR hcpcs = '{KNEE_MRI_CPT}')
                  AND payer_group IN ('BCBS', 'UnitedHealthcare', 'Aetna')
                  AND standard_charge_dollar IS NOT NULL
            )
            SELECT
                payer_group,
                COUNT(*) AS plan_count,
                MIN(standard_charge_dollar) AS min_rate,
                MEDIAN(standard_charge_dollar) AS median_rate,
                MAX(standard_charge_dollar) AS max_rate,
                ROUND(AVG(standard_charge_dollar), 2) AS avg_rate
            FROM distinct_rates
            GROUP BY payer_group
            ORDER BY payer_group
        """
        df = con.execute(query).fetchdf()
        df['hospital'] = hospital_name
        all_results.append(df)
        
        con.close()
        print(f"got {len(df)} payer groups")
    except Exception as e:
        print(f"ERROR: {e}")

combined = pd.concat(all_results, ignore_index=True)

combined = combined[['hospital', 'payer_group', 'plan_count', 
                     'min_rate', 'median_rate', 'max_rate', 'avg_rate']]
print(f"\nTotal rows across all hospitals: {len(combined)}")
combined

In [ ]:
output_path = Path("..") / "data" / "processed" / "day3_knee_mri_summary.csv"
combined.to_csv(output_path, index=False)
print(f"Saved {len(combined)} rows to {output_path}")
print(f"File size: {output_path.stat().st_size} bytes")